In [9]:
import pandas as pd
from utils import smiles2graph as smiles2graph_0
from utils import open_db, add_data_list, write
from functools import partial
from multiprocessing import Pool
from multiprocessing import cpu_count
from tqdm import tqdm
import yaml
from sklearn.model_selection import train_test_split

train_size = 0.9

# Get smiles2graph function
with open('smiles2graph_config.yaml', "r") as file:
    smiles2graph_config = yaml.safe_load(file)
smiles2graph = partial(smiles2graph_0, max_size = smiles2graph_config['max_size'], valid_atomic_nums = smiles2graph_config['valid_atomic_nums'], valid_bond_types = smiles2graph_config['valid_bond_types'])

# Open lmdb database
path = 'data/graphs/PUBCHEM'
env_train = open_db(path, 'train', mapsize=1099511627776, delete=True)
env_test = open_db(path, 'test', mapsize=1099511627776, delete=True)

# Load smiles csv (by chunks)
n_smiles_max = 100000  
n_threads = cpu_count() 
chunk_size = n_threads*1000
n_chunks = n_smiles_max // chunk_size + 1
chunk_iter = pd.read_csv("data/raw/PUBCHEM.csv", chunksize=chunk_size, sep='\t',index_col=0, header=None, nrows = n_smiles_max)

# main loop
for chunk in tqdm(chunk_iter, total=n_chunks, unit="chunk", desc="Processing smiles by chunks of size {}".format(chunk_size)):
    smiles_list = chunk.iloc[:, 0].values
    # Convert SMILES to graphs using smiles2graph + multiprocessing
    with Pool(processes=n_threads) as pool:
        graphs = list(pool.imap(smiles2graph, smiles_list))
    # Remove invalid graphs
    graphs = [g for g in graphs if g is not None]
    # Split graphs into train and test sets
    graphs_train, graphs_test = train_test_split(graphs, train_size=train_size, random_state=42)
    
    add_data_list(data_list = graphs_train, env = env_train)
    add_data_list(data_list = graphs_test, env = env_test)
    
# Close lmdb database
write(env_train, key = 'metadata', value = 'Some metadata')
write(env_test, key = 'metadata', value = 'Some metadata')
env_train.close()
env_test.close()



Processing smiles by chunks of size 28000: 100%|██████████| 4/4 [00:09<00:00,  2.27s/chunk]


In [10]:
from utils import LMDBDataset
dataset = LMDBDataset(path, split="train")
for data in dataset:
    print(data)
    break

{'node_labels': array([0, 2, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0], dtype=uint8), 'adjacency_matrix': <Compressed Sparse Row sparse matrix of dtype 'uint8'
	with 54 stored elements and shape (25, 25)>, 'edge_labels': <Compressed Sparse Row sparse matrix of dtype 'uint8'
	with 54 stored elements and shape (25, 25)>, 'SP_matrix': array([[ 0,  1,  2,  3,  4,  3,  2,  4,  5,  6,  6,  7,  8,  9, 10,  9,
         8,  7,  8,  9, 10,  9,  8,  7,  2],
       [ 1,  0,  1,  2,  3,  2,  1,  3,  4,  5,  5,  6,  7,  8,  9,  8,
         7,  6,  7,  8,  9,  8,  7,  6,  1],
       [ 2,  1,  0,  1,  2,  3,  2,  4,  5,  6,  6,  7,  8,  9, 10,  9,
         8,  7,  8,  9, 10,  9,  8,  7,  2],
       [ 3,  2,  1,  0,  1,  2,  3,  3,  4,  5,  5,  6,  7,  8,  9,  8,
         7,  6,  7,  8,  9,  8,  7,  6,  3],
       [ 4,  3,  2,  1,  0,  1,  2,  2,  3,  4,  4,  5,  6,  7,  8,  7,
         6,  5,  6,  7,  8,  7,  6,  5,  4],
       [ 3,  2,  3,  2,  1,  0,  1,  1,  2,  3,